In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention,191,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,intervention,191,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,2,intervention,191,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,intervention,191,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,3,intervention,191,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,other_causes,other_causes,95_plus,severe,3,baseline,45,0,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,45,0,0.0
539997,ylls,cause,other_causes,other_causes,95_plus,severe,4,baseline,45,0,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,45,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  581883.928509
                                  2                  462560.495008
                                  3                  408444.356236
                                  4                  369905.799251
                                  5                  343031.256007
intervention  maternal_disorders  1                  581883.928509
                                  2                  462560.495008
                                  3                  408444.356236
                                  4                  369905.799251
                                  5                  343031.256007
zero          maternal_disorders  1                  591306.980884
                                  2                  469119.806421
                                  3                  412121.233930
                                  4                  376933.536670
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,1,intervention,191,0,0.609244
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,intervention,191,0,0.000000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,intervention,191,0,0.000000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,intervention,191,0,0.000000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention,191,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,parturition,95_plus,severe,5,baseline,45,0,0.000000
1889996,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,45,0,0.000000
1889997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,45,0,0.000000
1889998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,45,0,0.000000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  106082.637923
                                  2                   84669.518932
                                  3                   68606.621111
                                  4                   57688.507031
                                  5                   43158.725305
              maternal_disorders  1                     161.489356
                                  2                     112.494489
                                  3                      91.323930
                                  4                      95.442412
                                  5                      97.580870
intervention  anemia              1                  106082.637923
                                  2                   84669.518932
                                  3                   68606.621111
                                  4                   57688.507031
            

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  106082.637923
                                  2                   84669.518932
                                  3                   68606.621111
                                  4                   57688.507031
                                  5                   43158.725305
              maternal_disorders  1                  582045.417865
                                  2                  462672.989497
                                  3                  408535.680166
                                  4                  370001.241664
                                  5                  343128.836878
intervention  anemia              1                  106082.637923
                                  2                   84669.518932
                                  3                   68606.621111
                                  4                   57688.507031
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,baseline,124,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,baseline,124,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,baseline,124,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,baseline,124,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,baseline,124,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
47995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,9,0,2420.918559
47996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,9,0,3773.579547
47997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,9,0,4052.344808
47998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,9,0,3255.511805


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.853957e+07
                      2                  1.533697e+07
                      3                  1.372313e+07
                      4                  1.287313e+07
                      5                  1.233084e+07
intervention  lbwsg   1                  1.853957e+07
                      2                  1.533697e+07
                      3                  1.372313e+07
                      4                  1.287313e+07
                      5                  1.233084e+07
zero          lbwsg   1                  1.858785e+07
                      2                  1.536768e+07
                      3                  1.374935e+07
                      4                  1.289434e+07
                      5                  1.233837e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1801.271863,zero
1,Female,0.0,0.019178,2,1419.783311,zero
2,Female,0.0,0.019178,3,1244.355229,zero
3,Female,0.0,0.019178,4,1059.997610,zero
4,Female,0.0,0.019178,5,769.541848,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,808.131007,intervention
746,Male,95.0,125.000000,2,825.228064,intervention
747,Male,95.0,125.000000,3,850.786851,intervention
748,Male,95.0,125.000000,4,878.164257,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.571411e+06
                      2                  1.524502e+06
                      3                  1.558915e+06
                      4                  1.480704e+06
                      5                  1.343512e+06
intervention  anemia  1                  1.571411e+06
                      2                  1.524502e+06
                      3                  1.558915e+06
                      4                  1.480704e+06
                      5                  1.343512e+06
zero          anemia  1                  1.690951e+06
                      2                  1.641059e+06
                      3                  1.664030e+06
                      4                  1.570185e+06
                      5                  1.390618e+06
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-511969.3565354217

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  592791.599547
                      2                  477444.392859
                      3                  438274.617550
                      4                  372755.685021
                      5                  289090.331032
intervention  anemia  1                  592791.599547
                      2                  477444.392859
                      3                  438274.617550
                      4                  372755.685021
                      5                  289090.331032
zero          anemia  1                  629672.544957
                      2                  508359.624334
                      3                  463846.788747
                      4                  391994.004109
                      5                  297836.660156
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-121352.99629493477

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  3.798496e+06
                      2                  3.418805e+06
                      3                  3.381926e+06
                      4                  3.124763e+06
                      5                  2.850846e+06
intervention  anemia  1                  3.798496e+06
                      2                  3.418805e+06
                      3                  3.381926e+06
                      4                  3.124763e+06
                      5                  2.850846e+06
zero          anemia  1                  4.098008e+06
                      2                  3.684232e+06
                      3                  3.612863e+06
                      4                  3.312747e+06
                      5                  2.946662e+06
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  363156.625935
                      2                  320810.978894
                      3                  288012.860341
                      4                  271581.933479
                      5                  231294.932434
baseline      ntd     1                  337550.298504
                      2                  301513.856527
                      3                  273366.783115
                      4                  259573.396157
                      5                  226710.744637
intervention  ntd     1                  191501.545763
                      2                  184065.447430
                      3                  178677.854520
                      4                  178412.378979
                      5                  189843.082529
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  3.904579e+06
                                  2                  3.503474e+06
                                  3                  3.450532e+06
                                  4                  3.182452e+06
                                  5                  2.894005e+06
              lbwsg               1                  1.853957e+07
                                  2                  1.533697e+07
                                  3                  1.372313e+07
                                  4                  1.287313e+07
                                  5                  1.233084e+07
              maternal_disorders  1                  5.820454e+05
                                  2                  4.626730e+05
                                  3                  4.085357e+05
                                  4                  3.700012e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)